# Benchmark Qwen2-VL-2B — 355 keyframe AIC

Chay lai benchmark tren toan bo anh de vuot nguong 100 anh ma de bai yeu cau
(hien `sample_results.json` moi co 79 anh cho model nay).

⚠️ **355 anh nay gom ca 290 anh da dung train QLoRA.** Voi bai nop thi vo hai —
model goc chua nap LoRA. Nhung KHONG dung tap nay de do truoc/sau khi nap LoRA:
phai dung 60 anh holdout rieng.


In [ ]:
import os

os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import torch

print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHONG CO')
assert torch.cuda.is_available(), 'Chua bat GPU'


In [ ]:
!pip install -q "transformers>=4.51,<5" accelerate bitsandbytes qwen-vl-utils
print('Cai xong')
import transformers
print('transformers:', transformers.__version__)


In [ ]:
import subprocess, sys
from pathlib import Path

# Clone tu fork public -- code luon khop ban moi nhat da day len.
REPO = 'https://github.com/lolizabrett-byte/Multimodal-Agentic-Retrieval-Engine.git'
NHANH = 'research/vlm-prompting'
DICH = Path('/kaggle/working/repo')

# Kaggle giu /kaggle/working giua cac version -> "clone neu chua co" se dung code
# cu cua lan chay truoc. Da mat mot luot GPU vi vay. Xoa roi clone lai moi lan.
import shutil
if DICH.exists():
    shutil.rmtree(DICH)
subprocess.run(['git', 'clone', '--depth', '1', '-b', NHANH, REPO, str(DICH)], check=True)

PKG = DICH / 'system1' / 'research' / 'vlm_prompting'
assert PKG.exists(), f'Khong thay code tai {PKG}'
sys.path.insert(0, str(PKG))

for ten in list(sys.modules):
    if ten.startswith(('vlm', 'benchmark_runner', 'checkpoint_utils', 'quality')):
        del sys.modules[ten]
print('Code tai:', PKG)

hash_code = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'],
                           cwd=DICH, capture_output=True, text=True).stdout.strip()
print('Commit:', hash_code)
assert hash_code, 'Khong doc duoc commit hash -- clone that bai'


In [ ]:
ANH_DIR = next(Path('/kaggle/input').glob('**/images'), None)
print('Thu muc anh:', ANH_DIR)
so_anh = len(list(ANH_DIR.glob('*.jpg')))
print('So anh:', so_anh)
assert so_anh >= 100, f'Chi co {so_anh} anh, de bai can >= 100'


In [ ]:
# --strict: model khong tai duoc thi NEM LOI, khong am tham roi ve mock.
# Goi qua subprocess (khong dung !shell) de duong dan Python noi suy dung.
lenh = [
    sys.executable, 'scripts/benchmark_runner.py',
    '--mode', 'mass',
    '--models', 'qwen2vl-2b,qwen25vl-3b',
    '--backend', 'transformers',
    '--strict', '--restart',
    '--frames-dir', str(ANH_DIR),
    '--out-dir', '/kaggle/working/ket_qua',
]
print('Chay:', ' '.join(lenh))
kq = subprocess.run(lenh, cwd=str(PKG))
assert kq.returncode == 0, f'benchmark_runner loi, ma thoat {kq.returncode}'


In [ ]:
import shutil

ZIP = shutil.make_archive('/kaggle/working/benchmark-355', 'zip', '/kaggle/working/ket_qua')
print('Da nen:', ZIP)
# Nen TRUOC cell kiem: kernel ERROR thi Kaggle khong luu /kaggle/working,
# mat sach ket qua du benchmark da chay xong. Da mat 1 luot GPU vi thu tu nguoc.


In [ ]:
import json
from collections import Counter

ra = Path('/kaggle/working/ket_qua')
print('File sinh ra:')
for f in sorted(ra.rglob('*')):
    if f.is_file():
        print(f'  {f.relative_to(ra)}  {f.stat().st_size:,} bytes')

kq_file = ra / 'sample_results.json'
assert kq_file.exists(), 'Khong thay sample_results.json'

d = json.loads(kq_file.read_text(encoding='utf-8'))
muc = d if isinstance(d, list) else d.get('results', d)
print()
print(f'Tong so muc: {len(muc)}')

# Dem theo TUNG model. Gop chung thi mot model chi ra vai anh van lot qua
# nguong 100 nho model kia bu vao.
MODEL_CHINH = 'qwen2vl-2b'
theo_model = {}
for mk in sorted({m['model'] for m in muc}):
    cua_no = [m for m in muc if m['model'] == mk]
    anh = {m['image'] for m in cua_no}
    # Truong that ten '_latency_sec' (co gach duoi) -- generate.py them tien to
    # '_' cho moi truong sieu du lieu. Doc nham ten khong-gach-duoi thi MOI muc
    # deu ra 0.0 va bao nham ca me la mock.
    gia = [m for m in cua_no if float(m.get('_latency_sec') or 0) == 0.0]
    en = [m for m in cua_no if m.get('caption_en') and m['caption_en'] != '[missing-en]']
    theo_model[mk] = (len(anh), len(gia), len(en), len(cua_no))
    print(f'  {mk:14s} {len(cua_no):4d} muc / {len(anh):4d} anh | mock: {len(gia):3d} '
          f'| co caption_en: {len(en)}/{len(cua_no)}')

# Model duoc chon phai dat nguong de bai va khong duoc co so mock.
anh_chinh, gia_chinh, _, _ = theo_model.get(MODEL_CHINH, (0, 0, 0, 0))
assert not gia_chinh, f'{MODEL_CHINH}: {gia_chinh} muc chay mock -- so lieu khong dung duoc'
assert anh_chinh >= 100, f'{MODEL_CHINH} chi {anh_chinh} anh -- chua dat nguong de bai'
print()
print(f'DAT: {MODEL_CHINH} co {anh_chinh} anh, khong co so gia.')

# Model phu hong thi bao ro, KHONG lam sap cell -- ket qua model chinh van phai giu.
for mk, (anh, gia, _, _) in theo_model.items():
    if mk == MODEL_CHINH:
        continue
    if gia or anh < 100:
        print(f'CANH BAO: {mk} chi {anh} anh, {gia} muc mock -- xem checkpoint_{mk}.json')


In [ ]:
# 13 ca loi cua Qwen2.5-VL-3B: chay rieng de LUU NGUYEN VAN output.
# Bao cao hien chi co thong bao loi ("khong tim thay JSON hop le, dai 320 ky tu"),
# khong co noi dung model that su sinh ra -> khong chan doan duoc, phai chay lai
# GPU moi biet. Cell nay lay bang chung truc tiep.
#
# KHONG nang MAX_NEW_TOKENS o day: gia thuyet "caption bi cat o 320 token" da bi
# bac bo bang 4 bang chung o phien truoc. Muc tieu la DOC output, khong phai sua mu.
ANH_LOI = ['003', '104', '111', '112', '118', '195', '216',
           '239', '240', '265', '269', '288', '342']

import tempfile, shutil as _sh

tmp_13 = Path(tempfile.mkdtemp(prefix='anh_loi_'))
thieu = []
for n in ANH_LOI:
    nguon = ANH_DIR / f'{n}.jpg'
    if nguon.exists():
        _sh.copy(nguon, tmp_13 / nguon.name)
    else:
        thieu.append(n)
print(f'Chep {len(list(tmp_13.glob("*.jpg")))}/13 anh vao {tmp_13}')
if thieu:
    print('THIEU:', thieu)

lenh_13 = [
    sys.executable, 'scripts/benchmark_runner.py',
    '--mode', 'mass',
    '--models', 'qwen25vl-3b',
    '--backend', 'transformers',
    '--strict', '--restart',
    '--frames-dir', str(tmp_13),
    '--out-dir', '/kaggle/working/ket_qua_13_ca_loi',
]
print('Chay:', ' '.join(lenh_13))
kq_13 = subprocess.run(lenh_13, cwd=str(PKG))
print('Ma thoat:', kq_13.returncode)

In [ ]:
import shutil

ZIP_13 = shutil.make_archive('/kaggle/working/ket-qua-13-ca-loi', 'zip',
                             '/kaggle/working/ket_qua_13_ca_loi')
print('Da nen:', ZIP_13)
# Nen TRUOC cell in: in nhieu text de lam kernel ERROR, mat sach /kaggle/working.

In [ ]:
import json

ck_13 = Path('/kaggle/working/ket_qua_13_ca_loi/checkpoint_qwen25vl-3b.json')
assert ck_13.exists(), f'Khong thay {ck_13}'

d13 = json.loads(ck_13.read_text(encoding='utf-8'))
xong13 = d13.get('da_xong') or {}
ok = sum(1 for v in xong13.values() if v.get('thanh_cong'))
print(f'{len(xong13)} anh | JSON hop le lan nay: {ok}/{len(xong13)}')
print('=' * 70)

# In NGUYEN VAN. Day la muc dich duy nhat cua cell nay -- doc thay model sinh ra
# gi thay vi doan.
for ten in sorted(xong13):
    v = xong13[ten]
    print(f'\n--- {ten} | thanh_cong={v.get("thanh_cong")} ---')
    if v.get('thanh_cong'):
        # Lan nay qua duoc -> prompt moi (phase 02) da sua duoc ca nay.
        print('  ket_qua:', json.dumps(v.get('ket_qua'), ensure_ascii=False)[:400])
        continue
    print('  loi     :', str(v.get('loi'))[:200])
    raw = v.get('raw_text')
    if raw:
        print(f'  raw_text ({len(raw)} ky tu):')
        print('  ' + raw.replace('\n', '\n  '))
    else:
        print('  raw_text: KHONG CO -- loi nay khong kem output tho')